# Structure Normalization

This notebook verifies the formal release and its reversible structural layer in PySpark.
It uses the completed structure-v2 snapshot when present, or builds the requested snapshot
from the formal release. The core run checks metadata and paragraph offsets, and measures
text reuse across the existing splits.

The overlap extension is written beside the full snapshot as structure-v2-overlap-v1. It
adds cross-split containment evidence from existing scores and length-pruned sketches. It
keeps the published snapshot and the assigned splits.

Set FOT_STRUCTURE_OUTPUT to a new directory to rebuild the core tables. Set
FOT_STRUCTURE_MODE=smoke to troubleshoot a small core run. Set FOT_STRUCTURE_FIGURES=1 to
render the lesson figures after running and saving Notebooks 01 and 02.


## Verify release inputs and initialize Spark

In [ ]:
import hashlib
import json
import os
import re
import sys
import tempfile
import time
import zlib
from datetime import UTC, datetime
from pathlib import Path

import duckdb
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/processed/corpus_releases/CURRENT_RELEASE.json").is_file()
)
PACKAGE = ROOT / "work_packages/formal_corpus_release_20261117"
METADATA = PACKAGE / "metadata_v1/corpus_metadata.duckdb"
META_RECEIPT = METADATA.with_name("METADATA_VERIFICATION.json")
POINTER = ROOT / "data/processed/corpus_releases/CURRENT_RELEASE.json"
release_pointer = json.loads(POINTER.read_text())
RELEASE = ROOT / release_pointer["catalog"]
RELEASE_RECEIPT = ROOT / release_pointer["receipt"]
release_receipt = json.loads(RELEASE_RECEIPT.read_text())
metadata_receipt = json.loads(META_RECEIPT.read_text())
FORMAL_RECORDS = release_receipt["formal_records"]
MODE = os.environ.get("FOT_STRUCTURE_MODE", "full")
if MODE not in {"smoke", "full"}:
    raise ValueError("FOT_STRUCTURE_MODE must be smoke or full.")
VERSION = "structure-v2"
default_output = f"{VERSION}-smoke" if MODE == "smoke" else VERSION
OUTPUT = Path(os.environ.get("FOT_STRUCTURE_OUTPUT", PACKAGE / default_output)).resolve()
if not OUTPUT.is_relative_to(PACKAGE) or not OUTPUT.parent.is_dir():
    raise ValueError("Choose an output directory under the release work package.")


def digest(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


def verify_digest(path, expected):
    if digest(path) != expected:
        raise ValueError(f"Input SHA-256 mismatch: {path.name}")


REBUILD_STRUCTURE = not OUTPUT.exists()
if not REBUILD_STRUCTURE:
    print(f"Using structure snapshot {OUTPUT.relative_to(ROOT)}.")
else:
    manifests = [
        RELEASE.parent / "BASELINE_MATERIALIZED.json",
        RELEASE.parent / "SUPPLEMENT_MATERIALIZED.json",
    ]
    parts = [entry for manifest in manifests for entry in json.loads(manifest.read_text())["parts"]]
    part_paths = [ROOT / entry["path"] for entry in parts]
    record_dir = RELEASE.parent / "records"
    if (
        any(path.parent != record_dir or path.is_symlink() for path in part_paths)
        or len(part_paths) != len(set(part_paths))
        or set(part_paths) != set(record_dir.glob("*.parquet"))
        or sum(entry["records"] for entry in parts) != release_receipt["records_materialized"]
    ):
        raise ValueError("The materialized record parts do not match the published inventories.")

    membership_file = RELEASE.parent / "RELEASE_MEMBERSHIP.parquet"
    passage_file = RELEASE.parent / "LONG_PASSAGE_FINGERPRINTS.parquet"
    bound_inputs = {
        METADATA: metadata_receipt["catalog_sha256"],
        RELEASE: release_receipt["catalog_sha256"],
        membership_file: release_receipt["membership_sha256"],
        passage_file: release_receipt["passage_fingerprint_sha256"],
        **{path: part["sha256"] for path, part in zip(part_paths, parts, strict=True)},
    }
    if (
        metadata_receipt["formal_release_version"] != release_pointer["version"]
        or release_receipt["version"] != release_pointer["version"]
        or release_pointer["formal_records"] != FORMAL_RECORDS
    ):
        raise ValueError("The formal release catalog, receipt, and metadata identities disagree.")
    input_hashes = {
        str(p.relative_to(ROOT)): digest(p)
        for p in (POINTER, RELEASE_RECEIPT, META_RECEIPT, *manifests)
    }
    for path, expected_hash in bound_inputs.items():
        verify_digest(path, expected_hash)
        input_hashes[str(path.relative_to(ROOT))] = expected_hash

    started = time.monotonic()
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError("Start this notebook in a fresh kernel to keep Spark settings isolated.")

    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault(
        "PYSPARK_SUBMIT_ARGS", f"--driver-memory {'2g' if MODE == 'smoke' else '4g'} pyspark-shell"
    )
    workers = int(os.environ.get("FOT_SPARK_WORKERS", "2" if MODE == "smoke" else "4"))
    spark = (
        SparkSession.builder.master(f"local[{workers}]")
        .appName(f"fear-temperature-structure-{MODE}")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
        .config("spark.sql.shuffle.partitions", str(workers * (4 if MODE == "smoke" else 12)))
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        .config("spark.sql.files.maxPartitionBytes", str(48 * 1024**2))
        .getOrCreate()
    )
    if (
        spark.sparkContext.appName != f"fear-temperature-structure-{MODE}"
        or spark.sparkContext.master != f"local[{workers}]"
    ):
        raise RuntimeError("Restart the kernel before starting a separate Spark run.")
    spark.sparkContext.setLogLevel("ERROR")
    workspace = tempfile.TemporaryDirectory(prefix=f".{VERSION}-", dir=OUTPUT.parent)
    STAGE = Path(workspace.name) / "snapshot"
    STAGE.mkdir()
    print(
        f"PySpark {spark.version} | {MODE} | {len(part_paths)} release parts verified", flush=True
    )

## Normalize metadata

Reproduce the published columns and check calendar and UTC edge cases. A source calendar day
may differ from the UTC day of its evidenced instant. A year range keeps a null single year.


In [ ]:
if REBUILD_STRUCTURE:
    # DuckDB is used only to transport the frozen formal metadata view.
    with tempfile.TemporaryDirectory(prefix="bridge-", dir=workspace.name) as bridge:
        with duckdb.connect(str(METADATA), read_only=True) as connection:
            connection.execute("SET enable_progress_bar = false")
            reader = connection.execute("SELECT * FROM formal_metadata").to_arrow_reader(25_000)
            for index, batch in enumerate(reader):
                target = Path(bridge) / f"{index:04d}.parquet"
                with pq.ParquetWriter(target, batch.schema, compression="zstd") as writer:
                    writer.write_batch(batch)

        source = spark.read.parquet(bridge)
        if source.count() != FORMAL_RECORDS:
            raise ValueError("The metadata view does not contain the expected formal population.")
        # Small positive controls from the archived structure-v1 audit.
        smoke_near_pairs = {(69_831, 69_834, False), (124_726, 148_308, True)}
        smoke_near_rows = {key for left, right, _ in smoke_near_pairs for key in (left, right)}
        if MODE == "smoke":
            # Include calendar, cleaning and near-duplicate evidence controls.
            source = source.where(
                (F.col("release_row") % 1021 == 0)
                | F.col("publication_year").isNull()
                | (F.year("publication_instant") != F.col("publication_year"))
                | F.col("release_row").isin(1729, 179823, 467977, *sorted(smoke_near_rows))
            )
        EXPECTED = source.count() if MODE == "smoke" else FORMAL_RECORDS

        def exact_utc(value):
            value = F.trim(value)
            explicit_zone = value.rlike(
                r"^\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(\.\d{1,6})?(Z|[+-]\d{2}:?\d{2})$"
            )
            colonized = F.regexp_replace(value, r"([+-]\d{2})(\d{2})$", "$1:$2")
            return F.when(explicit_zone, colonized.try_cast("timestamp"))

        def calendar_parts(value):
            raw = F.trim(value)
            day = raw.try_cast("date")
            month_start = F.concat(raw, F.lit("-01")).try_cast("date")
            is_day = raw.rlike(r"^\d{4}-\d{2}-\d{2}$") & day.isNotNull()
            is_month = raw.rlike(r"^\d{4}-\d{2}$") & month_start.isNotNull()
            is_year = raw.rlike(r"^\d{4}$")
            is_range = raw.rlike(r"^\d{4}[–/-]\d{4}$") & (
                F.substring(raw, 1, 4) <= F.substring(raw, -4, 4)
            )
            iso_source_day = F.substring(raw, 1, 10).try_cast("date")
            is_instant = exact_utc(raw).isNotNull() & iso_source_day.isNotNull()
            start = (
                F.when(is_day, day)
                .when(is_month, month_start)
                .when(
                    is_year | is_range,
                    F.concat(F.substring(raw, 1, 4), F.lit("-01-01")).try_cast("date"),
                )
                .when(is_instant, iso_source_day)
            )
            end = (
                F.when(is_day, day)
                .when(is_month, F.last_day(month_start))
                .when(
                    is_year | is_range,
                    F.concat(F.substring(raw, -4, 4), F.lit("-12-31")).try_cast("date"),
                )
                .when(is_instant, iso_source_day)
            )
            precision = (
                F.when(is_day, "day")
                .when(is_month, "month")
                .when(is_year, "year")
                .when(is_range, "year_range")
                .when(is_instant, "instant")
                .when(raw.isNull() | (raw == ""), "unknown")
                .otherwise("invalid")
            )
            year = F.when(is_day | is_month | is_year | is_instant, F.year(start)).cast("long")
            return precision, start, end, year

        cases = [
            ("2020-02-29", "day", "2020-02-29", "2020-02-29", 2020),
            ("2021-02-30", "invalid", None, None, None),
            ("2020-02", "month", "2020-02-01", "2020-02-29", 2020),
            ("2020", "year", "2020-01-01", "2020-12-31", 2020),
            ("2019–2020", "year_range", "2019-01-01", "2020-12-31", None),
            ("2020–2019", "invalid", None, None, None),
            ("2020-01-01T00:01:00+11:00", "instant", "2020-01-01", "2020-01-01", 2020),
            ("2020-01-01T00:01:00", "invalid", None, None, None),
            (None, "unknown", None, None, None),
        ]
        case_parts = calendar_parts(F.col("raw"))
        observed_cases = spark.createDataFrame([(c[0],) for c in cases], ["raw"]).select(
            "raw",
            case_parts[0].alias("precision"),
            case_parts[1].cast("string").alias("start"),
            case_parts[2].cast("string").alias("end"),
            case_parts[3].alias("year"),
        )
        if {tuple(r) for r in observed_cases.collect()} != set(cases):
            raise ValueError("Calendar boundary cases failed.")

        precision, start, end, year = calendar_parts(F.col("publication_raw"))
        raw_time = F.trim(F.col("publication_timestamp_raw"))
        publication_time = exact_utc(
            F.when(
                F.col("publication_timestamp_basis") == "native_publication_gmt_value_explicit_GMT",
                F.concat(raw_time, F.lit("Z")),
            ).otherwise(raw_time)
        )
        iso_pattern = "yyyy-MM-dd'T'HH:mm:ss.SSSSSS'Z'"
        transforms = {
            "publication_precision": precision,
            "publication_date_start": start,
            "publication_date_end": end,
            "publication_year": year,
            "publication_instant": publication_time,
            "publication_timestamp_utc": F.date_format(publication_time, iso_pattern),
            "ingested_at": exact_utc(F.col("ingested_at_raw")),
            "ingested_at_utc": F.date_format(exact_utc(F.col("ingested_at_raw")), iso_pattern),
            "source_retrieved_at": exact_utc(F.col("source_retrieved_at_raw")),
            "source_retrieved_at_utc": F.date_format(
                exact_utc(F.col("source_retrieved_at_raw")), iso_pattern
            ),
        }
        checks = (
            source.agg(
                F.count("*").alias("rows"),
                F.countDistinct("native_identity").alias("identities"),
                F.countDistinct("release_row").alias("release_keys"),
                F.count_if(
                    ~F.col("release_status").eqNullSafe(F.lit("formal"))
                    | F.col("format").isNull()
                    | F.col("format_basis").isNull()
                    | F.col("source_id").isNull()
                    | F.col("source_host").isNull()
                    | (F.col("source_host") == "No URL host recorded")
                ).alias("source_or_status_issues"),
                F.count_if(precision == "invalid").alias("invalid_calendar_values"),
                F.count_if((start.isNull() != end.isNull()) | (start > end)).alias(
                    "invalid_intervals"
                ),
                F.count_if(
                    F.col("publication_timestamp_basis").isNotNull() & publication_time.isNull()
                ).alias("unparsed_timestamp_evidence"),
                F.count_if(
                    ~exact_utc(F.date_format(publication_time, iso_pattern)).eqNullSafe(
                        publication_time
                    )
                ).alias("utc_round_trip_errors"),
                *[
                    F.count_if(~F.col(name).eqNullSafe(value)).alias(name)
                    for name, value in transforms.items()
                ],
            )
            .first()
            .asDict()
        )
        identities = ("rows", "identities", "release_keys")
        if [checks[key] for key in identities] != [EXPECTED] * 3 or any(
            value for key, value in checks.items() if key not in identities
        ):
            raise ValueError(f"Metadata contract failed: {checks}")

        utc_date = F.to_date("publication_instant")
        normalized = source.withColumns(transforms).withColumns(
            {
                "publication_instant_utc_date": utc_date,
                "publication_utc_date_differs": utc_date != F.col("publication_date_start"),
            }
        )
        normalized.write.mode("error").option("compression", "zstd").parquet(
            str(STAGE / "metadata")
        )

    meta = spark.read.parquet(str(STAGE / "metadata"))
    split_counts = {
        row.ml_split: row["count"] for row in meta.groupBy("ml_split").count().collect()
    }
    expected_splits = {
        name: release_receipt["counts"][f"split:{name}"] for name in ("train", "validation", "test")
    }
    if set(split_counts) != set(expected_splits) or (
        MODE == "full" and split_counts != expected_splits
    ):
        raise ValueError(f"Unexpected split counts: {split_counts}")
    date_counts = (
        meta.agg(
            F.count("publication_instant").alias("exact_publication_utc"),
            F.count_if("publication_utc_date_differs").alias("source_and_utc_day_differ"),
            F.count_if(F.year("publication_instant_utc_date") != F.col("publication_year")).alias(
                "source_and_utc_year_differ"
            ),
        )
        .first()
        .asDict()
    )
    print(f"Calendar contract: {len(cases)} boundary cases; {EXPECTED} metadata rows reconciled.")
    print(f"Source-calendar versus UTC differences: {date_counts}")

## Clean while segmenting by format

Separate explicit forum quote blocks and device signatures, or explicit article navigation. Record ambiguous markers for review. Preserve attributed article quotations, ordinary mentions, emoji, spelling, and all unaltered text. Group remaining paragraphs by natural blank-line boundaries.

In [ ]:
if REBUILD_STRUCTURE:
    # Integer offsets in this stage are zero-based Unicode code points with exclusive ends.
    FORUM_QUOTE = re.compile(
        r"(?i)\[(?P<bb_close>/)?quote(?:=[^\]\r\n]+)?\]"
        r"|<(?P<html_close>/)?blockquote\b[^<>]*>"
    )
    ARTICLE_NAV = re.compile(r"(?i)<(?P<nav_close>/)?nav\b[^<>]*>")
    VERBATIM = re.compile(r"(?i)(?:\[(?:code|pre)(?:=[^\]]+)?\]|<(?:code|pre)\b)")
    # cspell:ignore Tapatalk
    SIGNATURE = re.compile(
        r"(?im)^[ \t]*Sent from my (?:iPhone|iPad|[^\r\n]{1,65} using Tapatalk)"
        r"[.!]?[ \t]*(?:\r?\n[ \t]*)*\Z"
    )
    MD_QUOTE = re.compile(r"(?m)^[ \t]{0,3}>[ \t]+\S")
    REPLY_HEADER = re.compile(r"(?im)^.{1,90} said:[ \t]*\r?$")
    ARTICLE_CUE = re.compile(
        r"(?im)^[ \t]*(?:Read more|Related articles|Related posts|Advertisement)[ \t:]*\r?$"
    )
    PARAGRAPH_BREAK = re.compile(r"(?:\r?\n[ \t]*){2,}")
    LEXEME = re.compile(r"[^\W_]+", flags=re.UNICODE)
    ARTICLE_FORMATS = {"News", "Blog", "Article", "Community publication"}
    PASSAGE_WORDS = 100
    PASSAGE_CHARS = 500

    def text_sha(text):
        return hashlib.sha256(text.encode("utf-8")).hexdigest()

    def balanced_spans(text, pattern, families):
        stack = []
        spans = []
        for marker in pattern.finditer(text):
            kind = next(kind for kind in families if re.search(kind, marker.group(0), re.I))
            closing = any(value for value in marker.groupdict().values())
            if closing:
                if not stack or stack[-1][0] != kind:
                    return [], False
                _, start = stack.pop()
                if not stack:
                    spans.append((start, marker.end(), families[kind]))
            else:
                if marker.group(0).endswith("/>"):
                    return [], False
                stack.append((kind, marker.start()))
        return ([], False) if stack else (spans, True)

    def paragraph_offsets(text):
        start = 0
        spans = []
        for boundary in PARAGRAPH_BREAK.finditer(text):
            if text[start : boundary.start()].strip():
                spans.append((start, boundary.start()))
            start = boundary.end()
        if text[start:].strip():
            spans.append((start, len(text)))
        return spans

    def reshape(text, expected_sha, format_name):
        if text is None or expected_sha is None or text_sha(text) != expected_sha:
            raise ValueError("Release clean_text SHA binding failed")
        spans = []
        flags = []
        if format_name == "Forum":
            quotes, safe = balanced_spans(
                text,
                FORUM_QUOTE,
                {"blockquote": "forum_html_blockquote", r"\[/?quote": "forum_bbcode_quote"},
            )
            if safe and not (quotes and VERBATIM.search(text)):
                spans.extend(quotes)
            else:
                flags.append("forum_quote_markup_needs_review")
            signature = SIGNATURE.search(text)
            if signature and signature.start() > 0:
                if not any(
                    start < signature.end() and end > signature.start() for start, end, _ in spans
                ):
                    spans.append(
                        (signature.start(), signature.end(), "forum_tail_device_signature")
                    )
                else:
                    flags.append("forum_overlapping_signature_candidate")
            if MD_QUOTE.search(text) or REPLY_HEADER.search(text):
                flags.append("forum_plain_quote_marker_needs_review")
        elif format_name in ARTICLE_FORMATS:
            nav, safe = balanced_spans(text, ARTICLE_NAV, {"nav": "article_html_navigation"})
            if safe and not (nav and VERBATIM.search(text)):
                spans.extend(nav)
            else:
                flags.append("article_navigation_markup_needs_review")
            if ARTICLE_CUE.search(text):
                flags.append("article_plain_boilerplate_marker_needs_review")
        elif format_name == "Unresolved":
            flags.append("format_unresolved_structure_preserved")

        def cut(selected):
            pieces = []
            last = 0
            for start, end, rule in selected:
                if not last <= start < end <= len(text):
                    raise ValueError(f"Invalid {rule} source interval")
                pieces.extend((text[last:start], "\n\n"))
                last = end
            pieces.append(text[last:])
            return "".join(pieces)

        spans.sort()
        revised = cut(spans)
        if spans and not LEXEME.search(revised):
            spans = []
            revised = text
            flags.append("candidate_removal_would_leave_no_lexical_text")
        positions = paragraph_offsets(revised)
        removals = [(start, stop, rule, text_sha(text[start:stop])) for start, stop, rule in spans]
        return revised, text_sha(revised), removals, positions, sorted(set(flags))

    # Boundary regression checks before running any corpus rows.
    def example(text, family):
        return reshape(text, text_sha(text), family)

    assert example("[quote=Sam]old[/quote]\nI hate this heat 🔥 @Sam", "Forum")[0].endswith(
        "I hate this heat 🔥 @Sam"
    )
    assert example("[quote=Sam]old\nI hate this heat", "Forum")[2] == []
    assert example("> old\nI hate this heat", "Forum")[2] == []
    assert (
        example("A weather account.\nSent from my iPhone", "Forum")[2][0][2]
        == "forum_tail_device_signature"
    )
    assert (
        example("Hello\n<nav>Home / About</nav>\nI hate this heat", "News")[2][0][2]
        == "article_html_navigation"
    )
    assert example("Read more about fire warnings", "Blog")[0] == "Read more about fire warnings"
    assert example("<blockquote>Attributed interview statement</blockquote>", "News")[2] == []

    shape_type = T.StructType(
        [
            T.StructField("derived_text", T.StringType(), False),
            T.StructField("derived_sha256", T.StringType(), False),
            T.StructField(
                "removed_spans_in_release_clean_text",
                T.ArrayType(
                    T.StructType(
                        [
                            T.StructField("start", T.IntegerType(), False),
                            T.StructField("end", T.IntegerType(), False),
                            T.StructField("rule", T.StringType(), False),
                            T.StructField("removed_sha256", T.StringType(), False),
                        ]
                    ),
                    False,
                ),
                False,
            ),
            T.StructField(
                "paragraphs_in_derived_text",
                T.ArrayType(
                    T.StructType(
                        [
                            T.StructField("start", T.IntegerType(), False),
                            T.StructField("end", T.IntegerType(), False),
                        ]
                    ),
                    False,
                ),
                False,
            ),
            T.StructField("review_flags", T.ArrayType(T.StringType(), False), False),
        ]
    )
    shape_udf = F.udf(reshape, shape_type)
    keys = ["release_row", "native_identity", "body_sha256"]
    body = spark.read.parquet(*[str(path) for path in part_paths]).select(
        *keys, "clean_text", F.col("clean_sha256").alias("release_clean_sha256")
    )
    if MODE == "smoke":
        body = body.join(F.broadcast(meta.select(*keys)), keys, "left_semi")
    aligned = meta.select(*keys, "format", "source_host", "release_group_id", "ml_split").join(
        body, keys, "left"
    )
    (
        aligned.select(
            *keys,
            "format",
            "source_host",
            "release_group_id",
            "ml_split",
            "release_clean_sha256",
            shape_udf("clean_text", "release_clean_sha256", "format").alias("shape"),
        )
        .select("*", "shape.*")
        .drop("shape")
        .write.mode("error")
        .option("compression", "zstd")
        .parquet(str(STAGE / "paragraph_records"))
    )
    derived = spark.read.parquet(str(STAGE / "paragraph_records"))

    def valid_paragraphs(text, offsets):
        if not offsets:
            return False
        previous_end = 0
        for offset in offsets:
            if not previous_end <= offset.start < offset.end <= len(text):
                return False
            if not text[offset.start : offset.end].strip():
                return False
            previous_end = offset.end
        return True

    shape_checks = (
        derived.agg(
            F.count("*").alias("rows"),
            F.countDistinct("native_identity").alias("identities"),
            F.count_if(F.sha2("derived_text", 256) != F.col("derived_sha256")).alias(
                "sha_mismatches"
            ),
            F.count_if(F.size("paragraphs_in_derived_text") == 0).alias("no_paragraph"),
            F.count_if(
                ~F.udf(valid_paragraphs, "boolean")("derived_text", "paragraphs_in_derived_text")
            ).alias("invalid_paragraph_offsets"),
            F.count_if(
                (F.size("removed_spans_in_release_clean_text") > 0)
                != (F.col("release_clean_sha256") != F.col("derived_sha256"))
            ).alias("span_disagreements"),
        )
        .first()
        .asDict()
    )
    if shape_checks != {
        "rows": EXPECTED,
        "identities": EXPECTED,
        "sha_mismatches": 0,
        "no_paragraph": 0,
        "span_disagreements": 0,
        "invalid_paragraph_offsets": 0,
    }:
        raise ValueError(f"Paragraph integrity failed: {shape_checks}")
    print("Format-specific structure and paragraph offsets completed", shape_checks, flush=True)

## Audit structural edits and exact reuse

Replay edited spans against the hashed input. Compare release groups, native bodies, release bodies, derived bodies, and reused complete long paragraphs across the published splits.

In [ ]:
if REBUILD_STRUCTURE:
    # Reconstruct every edited record separately from recorded, hash-bound input spans.
    edited = derived.where(F.size("removed_spans_in_release_clean_text") > 0)
    review_edits = edited.join(body, keys, "left").select(
        "clean_text", "derived_text", "removed_spans_in_release_clean_text"
    )

    def valid_edit(before, after, spans):
        if before is None:
            return False
        last = 0
        fragments = []
        for span in spans:
            if not last <= span.start < span.end <= len(before):
                return False
            if text_sha(before[span.start : span.end]) != span.removed_sha256:
                return False
            fragments.append(before[last : span.start])
            last = span.end
        fragments.append(before[last:])
        return "\n\n".join(fragments) == after

    assert (
        review_edits.where(
            ~F.udf(valid_edit, "boolean")(
                "clean_text", "derived_text", "removed_spans_in_release_clean_text"
            )
        ).count()
        == 0
    )

    def passage_hashes(text, offsets):
        hashes = set()
        for position in offsets:
            paragraph = text[position.start : position.end]
            if len(paragraph) >= PASSAGE_CHARS:
                words = paragraph.split()
                if len(words) >= PASSAGE_WORDS:
                    hashes.add(text_sha(" ".join(words).casefold()))
        return sorted(hashes)

    def write_table(frame, name):
        frame.write.mode("error").option("compression", "zstd").parquet(str(STAGE / name))
        return spark.read.parquet(str(STAGE / name))

    passages = write_table(
        derived.select(
            "release_row",
            "native_identity",
            "ml_split",
            F.explode(
                F.udf(passage_hashes, "array<string>")("derived_text", "paragraphs_in_derived_text")
            ).alias("evidence_key"),
        ),
        "long_passage_members",
    )

    hash_fields = {
        "release_group": "release_group_id",
        "native_body": "body_sha256",
        "release_clean_body": "release_clean_sha256",
        "segment_input_body": "derived_sha256",
    }
    fingerprints = (
        derived.select(
            "release_row",
            "native_identity",
            "ml_split",
            F.explode(
                F.array(
                    *[
                        F.struct(
                            F.lit(kind).alias("evidence_type"), F.col(name).alias("evidence_key")
                        )
                        for kind, name in hash_fields.items()
                    ]
                )
            ).alias("item"),
        )
        .select("release_row", "native_identity", "ml_split", "item.*")
        .where(F.col("evidence_key").isNotNull())
    )
    fingerprints = fingerprints.unionByName(
        passages.withColumn("evidence_type", F.lit("long_passage"))
    )
    exact_groups = write_table(
        fingerprints.groupBy("evidence_type", "evidence_key")
        .agg(F.count("*").alias("records"), F.sort_array(F.collect_set("ml_split")).alias("splits"))
        .where(F.col("records") > 1),
        "exact_reuse_groups",
    )
    cross_exact = exact_groups.where(F.size("splits") > 1)
    exact_cross_members = write_table(
        fingerprints.join(
            cross_exact.select("evidence_type", "evidence_key"),
            ["evidence_type", "evidence_key"],
            "inner",
        ),
        "exact_cross_split_members",
    )
    exact_stats = {
        row.evidence_type: {
            "reused_keys": row.reused_keys,
            "extra_members": row.extra_members,
            "cross_split_keys": row.cross_split_keys,
        }
        for row in exact_groups.groupBy("evidence_type")
        .agg(
            F.count("*").alias("reused_keys"),
            F.sum(F.col("records") - 1).alias("extra_members"),
            F.count_if(F.size("splits") > 1).alias("cross_split_keys"),
        )
        .collect()
    }
    print("Exact-body, release-group and long-passage audit completed", exact_stats, flush=True)

## Generate near-duplicate candidates

Use deterministic bottom-k shingle samples with bounded posting buckets. Apply the cardinality upper bound for the exact Jaccard threshold.

In [ ]:
# Exact-identical derived bodies share a representative. The fuzzy index uses
# deterministic bottom-k (MinHash-style) samples of normalized five-word shingles.
# Exact Jaccard verification uses the strings themselves, not their index hashes.
WORD = re.compile(r"[^\W_]+(?:['’][^\W_]+)*", flags=re.UNICODE)
SHINGLE_WIDTH = 5
SAMPLE_SIZE = 12
MIN_SHINGLES = 30
MAX_BUCKET = 32
MIN_SHARED_SAMPLES = 3
JACCARD_PERCENT = 85


def shingles(text):
    words = WORD.findall(text.casefold())
    columns = [words[offset:] for offset in range(SHINGLE_WIDTH)]
    return {"\x1f".join(window) for window in zip(*columns, strict=False)}


def sketch(text):
    items = shingles(text)
    support = len(items)
    if support < MIN_SHINGLES:
        return support, []
    keys = sorted({zlib.crc32(item.encode("utf-8")) for item in items})
    return support, keys[:SAMPLE_SIZE]


def verify_pair(left, right):
    first = shingles(left)
    second = shingles(right)
    overlap = len(first & second)
    return overlap, len(first | second), min(len(first), len(second))


example_left = " ".join(f"word{i}" for i in range(70))
example_right = example_left + " added material"
a, b, support = verify_pair(example_left, example_right)
assert support >= MIN_SHINGLES and a * 100 >= b * JACCARD_PERCENT
assert verify_pair(example_left, "A completely different document")[0] == 0


if REBUILD_STRUCTURE:
    body_profiles = write_table(
        derived.groupBy("derived_sha256").agg(
            F.min("release_row").alias("representative"),
            F.count("*").alias("records"),
            F.sort_array(F.collect_set("ml_split")).alias("splits"),
        ),
        "candidate_body_profiles",
    )
    canonical = derived.join(
        body_profiles.select(F.col("representative").alias("release_row")), "release_row", "inner"
    )
    sketch_udf = F.udf(sketch, "struct<shingle_count:int,sample_keys:array<bigint>>")
    sketches = write_table(
        canonical.select(
            "release_row",
            sketch_udf("derived_text").alias("sketch"),
        ).select("release_row", "sketch.*"),
        "near_duplicate_sketches",
    )
    support_stats = (
        sketches.agg(
            F.count("*").alias("distinct_bodies"),
            F.count_if(F.col("shingle_count") >= MIN_SHINGLES).alias("fuzzy_indexed_bodies"),
            F.count_if(F.col("shingle_count") < MIN_SHINGLES).alias("below_fuzzy_evidence_minimum"),
        )
        .first()
        .asDict()
    )
    assert (
        sum(support_stats[k] for k in ("fuzzy_indexed_bodies", "below_fuzzy_evidence_minimum"))
        == support_stats["distinct_bodies"]
    )
    postings = sketches.select(
        "release_row", "shingle_count", F.explode("sample_keys").alias("sample")
    )
    posting_counts = write_table(postings.groupBy("sample").count(), "sample_bucket_counts")
    bucket_stats = (
        posting_counts.agg(
            F.count("*").alias("total_buckets"),
            F.count_if(F.col("count") > MAX_BUCKET).alias("suppressed_large_buckets"),
            F.coalesce(F.sum(F.when(F.col("count") > MAX_BUCKET, F.col("count"))), F.lit(0)).alias(
                "suppressed_postings"
            ),
            F.max("count").alias("largest_bucket"),
        )
        .first()
        .asDict()
    )
    print("Fuzzy fingerprints computed", support_stats, bucket_stats, flush=True)

    buckets = (
        postings.join(
            posting_counts.where(F.col("count").between(2, MAX_BUCKET)).select("sample"),
            "sample",
            "inner",
        )
        .groupBy("sample")
        .agg(F.collect_list(F.struct("release_row", "shingle_count")).alias("members"))
    )
    # Each shared bucket contributes one sample. The cardinality ratio filter cannot
    # remove a pair whose true Jaccard meets the threshold.
    potential = (
        buckets.select("sample", "members", F.explode("members").alias("a"))
        .select(
            "sample",
            "a",
            F.explode(
                F.filter(
                    "members",
                    lambda b: (
                        (F.col("a.release_row") < b.release_row)
                        & (
                            F.least(F.col("a.shingle_count"), b.shingle_count) * F.lit(100)
                            >= F.greatest(F.col("a.shingle_count"), b.shingle_count)
                            * F.lit(JACCARD_PERCENT)
                        )
                    ),
                )
            ).alias("b"),
        )
        .select(
            F.col("a.release_row").alias("left_row"),
            F.col("b.release_row").alias("right_row"),
            "sample",
        )
    )
    candidates = write_table(
        potential.groupBy("left_row", "right_row")
        .agg(F.sort_array(F.collect_set("sample")).alias("shared_samples"))
        .where(F.size("shared_samples") >= MIN_SHARED_SAMPLES),
        "near_duplicate_candidates",
    )
    candidate_pairs = candidates.count()
    print("Candidate pairs to verify", candidate_pairs, flush=True)

## Verify near duplicates and cross-split exposure

Score each candidate using exact shingle sets, then map the confirmed pairs back to every record represented by those bodies.

In [ ]:
if REBUILD_STRUCTURE:

    def side(frame, prefix):
        return frame.select(
            F.col("release_row").alias(f"{prefix}_row"),
            F.col("derived_text").alias(f"{prefix}_text"),
            F.col("native_identity").alias(f"{prefix}_native_identity"),
            F.col("derived_sha256").alias(f"{prefix}_body_sha256"),
            F.col("format").alias(f"{prefix}_format"),
            F.col("source_host").alias(f"{prefix}_source_host"),
        )

    score_udf = F.udf(verify_pair, "struct<intersection:int,union_size:int,smaller_set:int>")
    scored = (
        candidates.join(side(canonical, "left"), "left_row")
        .join(side(canonical, "right"), "right_row")
        .withColumn("score", score_udf("left_text", "right_text"))
        .select("*", "score.*")
        .drop("score", "left_text", "right_text")
        .withColumn("jaccard", F.col("intersection") / F.col("union_size"))
        .withColumn("containment", F.col("intersection") / F.col("smaller_set"))
        .withColumn(
            "meets_near_duplicate_rule",
            (F.col("intersection") >= MIN_SHINGLES)
            & (F.col("intersection") * 100 >= F.col("union_size") * JACCARD_PERCENT),
        )
    )
    scored = write_table(scored, "near_duplicate_pair_evidence")
    assert scored.count() == candidate_pairs

    def profile_side(prefix):
        return body_profiles.select(
            F.col("representative").alias(f"{prefix}_row"),
            F.col("splits").alias(f"{prefix}_splits"),
            F.col("records").alias(f"{prefix}_exact_body_records"),
        )

    confirmed = write_table(
        scored.where("meets_near_duplicate_rule")
        .join(profile_side("left"), "left_row")
        .join(profile_side("right"), "right_row")
        .withColumn("cross_split", F.size(F.array_union("left_splits", "right_splits")) > 1),
        "near_duplicate_matches",
    )
    near_counts = (
        confirmed.agg(
            F.count("*").alias("matched_pairs"),
            F.count_if("cross_split").alias("cross_split_pairs"),
        )
        .first()
        .asDict()
    )
    smoke_near_controls = None
    if MODE == "smoke":
        control_matches = confirmed.where(
            F.col("left_row").isin(*sorted(smoke_near_rows))
            & F.col("right_row").isin(*sorted(smoke_near_rows))
        ).select("left_row", "right_row", "cross_split")
        if not smoke_near_pairs.issubset({tuple(row) for row in control_matches.collect()}):
            raise ValueError("Known same-split and cross-split near-duplicate controls failed.")
        smoke_near_controls = len(smoke_near_pairs)

    cross_pairs = confirmed.where("cross_split")

    def endpoints(frame):
        return (
            frame.select(F.col("left_body_sha256").alias("derived_sha256"))
            .union(frame.select("right_body_sha256"))
            .distinct()
        )

    near_record_count = derived.join(endpoints(confirmed), "derived_sha256", "left_semi").count()
    cross_near = derived.join(endpoints(cross_pairs), "derived_sha256", "left_semi").select(
        "release_row",
        "native_identity",
        "ml_split",
        F.lit("near_duplicate").alias("evidence_type"),
        F.col("derived_sha256").alias("evidence_key"),
    )
    leak_members = write_table(
        exact_cross_members.unionByName(cross_near), "leakage_review_records"
    )
    print("Near-duplicate verification complete", near_counts, flush=True)

## Publish evidence and actual counts

Report rule counts, edge counts and distinct affected records. Publish only after integrity checks and leave the official release and split memberships intact.

In [ ]:
if REBUILD_STRUCTURE:
    formats = (
        derived.groupBy("format")
        .agg(
            F.count("*").alias("records"),
            F.count_if(F.size("removed_spans_in_release_clean_text") > 0).alias("edited_records"),
            F.sum(F.size("paragraphs_in_derived_text")).alias("paragraphs"),
            F.count_if(F.size("review_flags") > 0).alias("records_with_review_flags"),
        )
        .orderBy("format")
    )
    removal_rules = (
        derived.select(
            "native_identity", F.explode("removed_spans_in_release_clean_text").alias("span")
        )
        .select("native_identity", "span.*")
        .groupBy("rule")
        .agg(
            F.count("*").alias("removed_spans"),
            F.countDistinct("native_identity").alias("edited_records"),
            F.sum(F.col("end") - F.col("start")).alias("removed_codepoints"),
        )
        .orderBy("rule")
    )
    review_flags = (
        derived.select(F.explode("review_flags").alias("reason"))
        .groupBy("reason")
        .count()
        .orderBy("reason")
    )

    def collect_dicts(frame):
        return [row.asDict(recursive=True) for row in frame.collect()]

    format_rows = collect_dicts(formats)
    assert sum(row["records"] for row in format_rows) == EXPECTED
    summary = {
        "version": VERSION,
        "completed_at_utc": datetime.now(UTC).isoformat(),
        "input_release_version": release_pointer["version"],
        "input_hashes": input_hashes,
        "pyspark_version": spark.version,
        "metadata_checks": checks,
        "selected_split_counts": split_counts,
        "normalization": {
            "precision_counts": {
                r.publication_precision: r["count"]
                for r in meta.groupBy("publication_precision").count().collect()
            },
            "exact_publication_utc": meta.where(
                F.col("publication_timestamp_utc").isNotNull()
            ).count(),
            "format_evidence": (
                "Copied from metadata v1 with its format_basis; unresolved values retained."
            ),
        },
        "paragraph_integrity": shape_checks,
        "total_paragraphs": sum(row["paragraphs"] for row in format_rows),
        "edited_records": sum(row["edited_records"] for row in format_rows),
        "records_with_review_flags": sum(row["records_with_review_flags"] for row in format_rows),
        "cleaning_by_format": format_rows,
        "cleaning_by_rule": collect_dicts(removal_rules),
        "cleaning_review_flags": collect_dicts(review_flags),
        "long_passage_memberships": passages.count(),
        "exact_reuse": exact_stats,
        "fuzzy_search": {
            "shingle_width": SHINGLE_WIDTH,
            "min_unique_shingles": MIN_SHINGLES,
            "sample_size": SAMPLE_SIZE,
            "max_bucket": MAX_BUCKET,
            "min_shared_samples": MIN_SHARED_SAMPLES,
            "min_jaccard": JACCARD_PERCENT / 100,
            **support_stats,
            **bucket_stats,
            "candidate_pairs_verified": candidate_pairs,
            **near_counts,
            "records_in_matched_pairs": near_record_count,
            "cross_split_affected_records": cross_near.count(),
            "coverage": "Approximate search suppresses large buckets. Fuzzy matching "
            "requires minimum shingle evidence. The Jaccard rule omits containment-only copying. "
            "Exact checks include all selected records.",
        },
        "unique_records_in_leakage_review": leak_members.select("native_identity")
        .distinct()
        .count(),
        "leakage_review_by_split": collect_dicts(
            leak_members.select("native_identity", "ml_split")
            .distinct()
            .groupBy("ml_split")
            .count()
            .orderBy("ml_split")
        ),
        "original_text_source": "Native body referenced by native_identity and body_sha256, "
        "and immutable release clean_text referenced by release_clean_sha256.",
        "split_changes_applied": 0,
        "elapsed_seconds": round(time.monotonic() - started, 1),
    }

    # Measure the published fingerprint inventory on its two populations.
    release_members = spark.read.parquet(str(membership_file)).select(
        "release_row", "formal_eligible"
    )
    release_passage_counts = (
        spark.read.parquet(str(passage_file))
        .join(release_members, "release_row")
        .groupBy("passage_sha256")
        .agg(F.count("*").alias("all_rows"), F.count_if("formal_eligible").alias("formal_rows"))
        .agg(
            F.sum("all_rows").alias("materialized_instances"),
            F.count_if(F.col("all_rows") > 1).alias("materialized_reused_hashes"),
            F.sum(F.greatest(F.col("all_rows") - 1, F.lit(0))).alias("materialized_extra_members"),
            F.sum("formal_rows").alias("formal_instances"),
            F.count_if(F.col("formal_rows") > 1).alias("formal_reused_hashes"),
            F.sum(F.greatest(F.col("formal_rows") - 1, F.lit(0))).alias("formal_extra_members"),
        )
        .first()
        .asDict()
    )
    if (
        release_passage_counts["materialized_instances"]
        != release_receipt["long_passage_instances"]
        or release_passage_counts["materialized_reused_hashes"]
        != release_receipt["reused_long_passage_hashes"]
    ):
        raise ValueError("Published materialized passage counts do not reconcile with the receipt.")

    # Schema and exact file bytes are verifiable independently of a Spark session.
    tables = {}
    for table in sorted(folder for folder in STAGE.iterdir() if folder.is_dir()):
        files = sorted(table.glob("*.parquet"))
        if not files or not (table / "_SUCCESS").is_file():
            raise ValueError(f"Incomplete staged output: {table.name}")
        entries = [
            {
                "path": file.relative_to(STAGE).as_posix(),
                "rows": pq.ParquetFile(file).metadata.num_rows,
                "sha256": digest(file),
            }
            for file in files
        ]
        tables[table.name] = {
            "rows": sum(entry["rows"] for entry in entries),
            "spark_schema": spark.read.parquet(str(table)).schema.jsonValue(),
            "files": entries,
        }
    if tables["metadata"]["rows"] != EXPECTED or tables["paragraph_records"]["rows"] != EXPECTED:
        raise ValueError(
            "Published metadata and paragraph tables must have one row per selected record."
        )

    manifest = {
        "schema_version": VERSION,
        "run_mode": MODE,
        "input_release_version": release_pointer["version"],
        "input_release_files_sha256": input_hashes,
        "tables": tables,
        "calendar": {
            "publication_date_start/end": "source calendar dates at their stated precision",
            "publication_timestamp_utc": "UTC instant when present in the source evidence",
            "year_ranges": "null publication_year",
        },
        "offsets": {
            "unit": "zero-based, half-open Unicode code points",
            "removed_spans_in_release_clean_text": "clean_text identified by release_clean_sha256",
            "paragraphs_in_derived_text": "derived_text identified by derived_sha256",
        },
        "spark": {
            "version": spark.version,
            "python": sys.version.split()[0],
            "master": spark.sparkContext.master,
            "time_zone": spark.conf.get("spark.sql.session.timeZone"),
            "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
            "timestamp_encoding": spark.conf.get("spark.sql.parquet.outputTimestampType"),
        },
    }
    for relative, expected_hash in input_hashes.items():
        verify_digest(ROOT / relative, expected_hash)
    (STAGE / "MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    summary.update(
        run_mode=MODE,
        selected_records=EXPECTED,
        source_formal_records=FORMAL_RECORDS,
        immutable_release_parts_verified=len(part_paths),
        calendar_boundary_cases_passed=len(cases),
        smoke_near_duplicate_controls_passed=smoke_near_controls,
        date_semantics=date_counts,
        published_passage_reconciliation=release_passage_counts,
        manifest_sha256=digest(STAGE / "MANIFEST.json"),
    )
    # The manifest already records every input hash. Keep the audit summary readable in a notebook.
    summary.pop("input_hashes", None)
    summary["elapsed_seconds"] = round(time.monotonic() - started, 1)
    (STAGE / "STRUCTURE_AUDIT.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    (STAGE / "README.md").write_text(
        f"""# Structure normalization v2 ({MODE})

    Source: {FORMAL_RECORDS:,} formal release records. This run selected {EXPECTED:,} records.
    See `MANIFEST.json` for inputs, table schemas, row counts and file hashes, and
    `STRUCTURE_AUDIT.json` for observed checks. Smoke results describe the selected rows.

    - `metadata` reproduces the published columns and validates date and UTC invariants.
      Keep source calendar dates separate from exact UTC dates. Date ranges retain null years.
    - `paragraph_records.derived_text` is the derived text.
      `removed_spans_in_release_clean_text` locates removals in the frozen release clean text,
      while `paragraphs_in_derived_text` locates paragraphs in the derived text.
    - Duplicate tables contain scoped evidence. The fuzzy candidate search applies the existing
      Jaccard rule and does not cover containment-only copying, all short quotations or paraphrases.
    - The split labels are inherited. No records are removed, promoted or reassigned.

    Published paragraph-fingerprint counts and this notebook's paragraph counts use their
    respective populations and rules.
    """,
        encoding="utf-8",
    )
    spark.stop()
    if OUTPUT.exists():
        raise FileExistsError(f"The destination appeared during the run: {OUTPUT.name}")
    STAGE.rename(OUTPUT)
    workspace.cleanup()
    print(
        json.dumps(
            {
                "output": OUTPUT.relative_to(ROOT).as_posix(),
                "mode": MODE,
                "selected_records": EXPECTED,
                "tables_verified": len(tables),
                "cross_split_pairs": summary["fuzzy_search"]["cross_split_pairs"],
                "published_passages": release_passage_counts,
            },
            indent=2,
        ),
        flush=True,
    )

## Extend the cross-split evidence

Read the manifest-bound core snapshot. Keep the published Jaccard rule and add containment
candidates with at least 30 shared five-word shingles and at least 90% containment. Score
length-pruned pairs reachable through the published sketches separately from already-scored
candidates. Record ordered word overlap where measured. This search retains the published
bucket and sketch limits. Pair evidence remains distinct from the record decision. The compact
records table also marks members of the same formal release group for review.


In [ ]:
from difflib import SequenceMatcher

# The formal structure is immutable. Publish a small, independently versioned evidence layer.
if MODE == "full":
    source_manifest = json.loads((OUTPUT / "MANIFEST.json").read_text())
    source_audit = json.loads((OUTPUT / "STRUCTURE_AUDIT.json").read_text())
    manifest_hash = digest(OUTPUT / "MANIFEST.json")
    if (
        source_audit["manifest_sha256"] != manifest_hash
        or source_manifest["run_mode"] != "full"
        or source_audit["selected_records"] != FORMAL_RECORDS
    ):
        raise ValueError("A verified full structure snapshot is required for overlap evidence.")

    def read_bound_tables(directory, entries):
        for name, parts in entries.items():
            files = sorted((directory / name).glob("*.parquet"))
            if not files or {item["path"] for item in parts} != {
                file.relative_to(directory).as_posix() for file in files
            }:
                raise ValueError(f"Unexpected Parquet inventory for {name}.")
            for item in parts:
                verify_digest(directory / item["path"], item["sha256"])

    EVIDENCE = OUTPUT.with_name(f"{OUTPUT.name}-overlap-v1")
    required = (
        "paragraph_records",
        "candidate_body_profiles",
        "near_duplicate_pair_evidence",
        "near_duplicate_sketches",
        "sample_bucket_counts",
    )
    bound_parts = {name: source_manifest["tables"][name]["files"] for name in required}
    read_bound_tables(OUTPUT, bound_parts)
    if EVIDENCE.exists():
        evidence_manifest = json.loads((EVIDENCE / "MANIFEST.json").read_text())
        if evidence_manifest["structure_manifest_sha256"] != manifest_hash:
            raise ValueError("The overlap evidence belongs to a different structure snapshot.")
        verify_digest(EVIDENCE / "AUDIT.json", evidence_manifest["audit_sha256"])
        read_bound_tables(EVIDENCE, evidence_manifest["tables"])
        print(f"Verified existing overlap evidence at {EVIDENCE.relative_to(ROOT)}.")
    else:
        overlap_started = time.monotonic()
        os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
        os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
        overlap_spark = (
            SparkSession.builder.master("local[3]")
            .appName("fear-temperature-overlap")
            .config("spark.driver.memory", "3g")
            .config("spark.driver.host", "127.0.0.1")
            .config("spark.driver.bindAddress", "127.0.0.1")
            .config("spark.ui.enabled", "false")
            .config("spark.ui.showConsoleProgress", "false")
            .config("spark.sql.shuffle.partitions", "24")
            .config("spark.sql.adaptive.enabled", "true")
            .getOrCreate()
        )
        overlap_spark.sparkContext.setLogLevel("ERROR")

        def overlap_metrics(left, right, detail):
            shared, total, smaller = verify_pair(left, right)
            if not detail and (shared < MIN_SHINGLES or shared * 10 < smaller * 9):
                return shared, total, smaller, None, None, None, None
            first, second = sorted(
                (WORD.findall(left.casefold()), WORD.findall(right.casefold())), key=len
            )
            # Bound the work on exceptionally long documents. Null denotes unmeasured alignment.
            if len(first) * len(second) > 24_000_000:
                return shared, total, smaller, len(first), None, None, None
            blocks = SequenceMatcher(None, first, second, autojunk=False).get_matching_blocks()
            return (
                shared,
                total,
                smaller,
                len(first),
                max(m.size for m in blocks),
                sum(m.size for m in blocks if m.size >= 20),
                "ordered_token_alignment",
            )

        metrics_type = (
            "struct<intersection:int,union_size:int,smaller_set:int,shorter_words:int,"
            "longest_aligned_run_words:int,aligned_words_in_runs_of_20:int,alignment_method:string>"
        )
        measure = F.udf(overlap_metrics, metrics_type, useArrow=False)
        try:
            with tempfile.TemporaryDirectory(prefix=".overlap-", dir=OUTPUT.parent) as local:
                staged = Path(local) / "snapshot"
                paragraphs = overlap_spark.read.parquet(str(OUTPUT / "paragraph_records"))
                profiles = overlap_spark.read.parquet(str(OUTPUT / "candidate_body_profiles"))
                scored_pairs = overlap_spark.read.parquet(
                    str(OUTPUT / "near_duplicate_pair_evidence")
                )

                def pair_profiles(frame):
                    for side_name in ("left", "right"):
                        frame = frame.join(
                            profiles.select(
                                F.col("representative").alias(f"{side_name}_row"),
                                F.col("derived_sha256").alias(f"{side_name}_sha"),
                                F.col("splits").alias(f"{side_name}_splits"),
                            ),
                            f"{side_name}_row",
                        )
                    return frame.where(F.size(F.array_union("left_splits", "right_splits")) > 1)

                strict = pair_profiles(scored_pairs.where("meets_near_duplicate_rule"))
                within = pair_profiles(
                    scored_pairs.where(
                        (F.col("intersection") >= MIN_SHINGLES)
                        & (F.col("intersection") * 10 >= F.col("smaller_set") * 9)
                        & ~F.col("meets_near_duplicate_rule")
                    )
                )
                key_columns = (
                    "left_row",
                    "right_row",
                    "left_sha",
                    "right_sha",
                    "left_splits",
                    "right_splits",
                )
                count_columns = ("intersection", "union_size", "smaller_set")
                alignment_columns = (
                    "shorter_words",
                    "longest_aligned_run_words",
                    "aligned_words_in_runs_of_20",
                )
                strict = strict.select(
                    *key_columns,
                    *count_columns,
                    *[F.lit(None).cast("int").alias(key) for key in alignment_columns],
                    F.lit(None).cast("string").alias("alignment_method"),
                    F.lit("strict_jaccard").alias("rule"),
                    F.lit("published_scored_candidates").alias("candidate_path"),
                )

                indexed = overlap_spark.read.parquet(str(OUTPUT / "near_duplicate_sketches"))
                counts = overlap_spark.read.parquet(str(OUTPUT / "sample_bucket_counts"))
                allowed = counts.where(F.col("count").between(2, MAX_BUCKET)).select("sample")
                postings = indexed.select(
                    "release_row", "shingle_count", F.explode("sample_keys").alias("sample")
                )
                members = (
                    postings.join(F.broadcast(allowed), "sample")
                    .groupBy("sample")
                    .agg(F.collect_list(F.struct("release_row", "shingle_count")).alias("members"))
                )
                asymmetric = (
                    members.select("members", F.explode("members").alias("a"))
                    .select(
                        "a",
                        F.explode(
                            F.filter(
                                "members",
                                lambda b: (
                                    (F.col("a.release_row") < b.release_row)
                                    & (
                                        F.least(F.col("a.shingle_count"), b.shingle_count) * 100
                                        < F.greatest(F.col("a.shingle_count"), b.shingle_count)
                                        * JACCARD_PERCENT
                                    )
                                ),
                            )
                        ).alias("b"),
                    )
                    .select(
                        F.col("a.release_row").alias("left_row"),
                        F.col("b.release_row").alias("right_row"),
                    )
                    .groupBy("left_row", "right_row")
                    .count()
                    .where(F.col("count") >= MIN_SHARED_SAMPLES)
                    .drop("count")
                )
                # Check only cross-split pairs. Save the narrow roster before reading any text.
                scratch = Path(local) / "asymmetric"
                pair_profiles(asymmetric).write.mode("errorifexists").parquet(str(scratch))
                extra = overlap_spark.read.parquet(str(scratch))
                extra_n = extra.count()
                print(f"Cross-split length-pruned candidates to verify: {extra_n}", flush=True)
                to_measure = (
                    within.select(*key_columns)
                    .withColumn("candidate_path", F.lit("published_scored_candidates"))
                    .unionByName(
                        extra.select(*key_columns).withColumn(
                            "candidate_path", F.lit("length_pruned_sketches")
                        )
                    )
                )
                ids = (
                    to_measure.select(F.col("left_row").alias("release_row"))
                    .union(to_measure.select("right_row"))
                    .distinct()
                )
                text_path = Path(local) / "endpoint_text"
                paragraphs.select("release_row", "derived_text").join(
                    F.broadcast(ids), "release_row", "left_semi"
                ).write.mode("errorifexists").option("compression", "zstd").parquet(str(text_path))
                texts = overlap_spark.read.parquet(str(text_path))
                for side_name in ("left", "right"):
                    to_measure = to_measure.join(
                        texts.select(
                            F.col("release_row").alias(f"{side_name}_row"),
                            F.col("derived_text").alias(f"{side_name}_text"),
                        ),
                        f"{side_name}_row",
                    )
                containment = (
                    to_measure.withColumn(
                        "measures",
                        measure(
                            "left_text",
                            "right_text",
                            F.col("candidate_path") == "published_scored_candidates",
                        ),
                    )
                    .select(*key_columns, "measures.*", "candidate_path")
                    .where(
                        (F.col("intersection") >= MIN_SHINGLES)
                        & (F.col("intersection") * 10 >= F.col("smaller_set") * 9)
                    )
                    .withColumn("rule", F.lit("containment_candidate"))
                )
                pairs = (
                    strict.unionByName(containment)
                    .withColumn(
                        "pair_id",
                        F.sha2(
                            F.concat_ws(
                                ":",
                                F.least("left_sha", "right_sha"),
                                F.greatest("left_sha", "right_sha"),
                            ),
                            256,
                        ),
                    )
                    .withColumn("jaccard", F.col("intersection") / F.col("union_size"))
                    .withColumn("containment", F.col("intersection") / F.col("smaller_set"))
                )
                pairs.write.mode("errorifexists").option("compression", "zstd").parquet(
                    str(staged / "pairs")
                )
                pairs = overlap_spark.read.parquet(str(staged / "pairs"))
                pair_counts = [
                    row.asDict()
                    for row in pairs.groupBy("rule", "candidate_path").count().collect()
                ]
                strict_count = sum(
                    row["count"] for row in pair_counts if row["rule"] == "strict_jaccard"
                )
                if strict_count != source_audit["fuzzy_search"]["cross_split_pairs"]:
                    raise ValueError("The original strict overlap evidence did not reconcile.")
                endpoints_e = pairs.select(
                    F.col("left_sha").alias("derived_sha256"), "pair_id", "rule"
                ).union(pairs.select(F.col("right_sha").alias("derived_sha256"), "pair_id", "rule"))
                direct_by_sha = endpoints_e.groupBy("derived_sha256").agg(
                    F.sort_array(F.collect_set("pair_id")).alias("overlap_pair_ids"),
                    F.sort_array(F.collect_set("rule")).alias("overlap_rules"),
                )
                parent_keys = paragraphs.select(
                    "release_row",
                    "native_identity",
                    "derived_sha256",
                    "release_group_id",
                    "ml_split",
                )
                matches = parent_keys.join(F.broadcast(direct_by_sha), "derived_sha256")
                direct_count = matches.count()
                groups = matches.select("release_group_id").distinct()
                members_e = (
                    parent_keys.join(F.broadcast(groups), "release_group_id", "left_semi")
                    .join(F.broadcast(direct_by_sha), "derived_sha256", "left")
                    .withColumn(
                        "overlap_status",
                        F.when(
                            F.col("overlap_pair_ids").isNotNull(), F.lit("direct_evidence")
                        ).otherwise(F.lit("related_release_group")),
                    )
                )
                members_e.write.mode("errorifexists").option("compression", "zstd").parquet(
                    str(staged / "records")
                )
                tables = {
                    name: [
                        {
                            "path": file.relative_to(staged).as_posix(),
                            "sha256": digest(file),
                            "rows": pq.ParquetFile(file).metadata.num_rows,
                        }
                        for file in sorted((staged / name).glob("*.parquet"))
                    ]
                    for name in ("pairs", "records")
                }
                summary_e = {
                    "source_records": FORMAL_RECORDS,
                    "pairs": pair_counts,
                    "direct_records": direct_count,
                    "direct_or_related_group_records": sum(
                        item["rows"] for item in tables["records"]
                    ),
                    "length_pruned_cross_split_candidates_verified": extra_n,
                    "setting": {
                        "containment_threshold": 0.90,
                        "min_shared_shingles": MIN_SHINGLES,
                        "max_bucket": MAX_BUCKET,
                        "minimum_shared_sketch_samples": MIN_SHARED_SAMPLES,
                        "scope": "Cross-split candidates from published bottom-k sketches",
                    },
                    "elapsed_seconds": round(time.monotonic() - overlap_started, 1),
                }
                audit_path = staged / "AUDIT.json"
                audit_path.write_text(json.dumps(summary_e, indent=2) + "\n")
                manifest_e = {
                    "schema_version": 1,
                    "structure_manifest_sha256": manifest_hash,
                    "audit_sha256": digest(audit_path),
                    "tables": tables,
                }
                (staged / "MANIFEST.json").write_text(json.dumps(manifest_e, indent=2) + "\n")
                if digest(OUTPUT / "MANIFEST.json") != manifest_hash or EVIDENCE.exists():
                    raise RuntimeError("Overlap input or output changed during publication.")
                staged.rename(EVIDENCE)
                print(json.dumps(summary_e, indent=2), flush=True)
        finally:
            overlap_spark.stop()

## Archived v1 index assessment

These observations describe the earlier full v1 snapshot. They do not describe the smoke run. They do not change the formal
catalog, the Parquet layout, or the audit thresholds.

- **Structured filtering.** The five metadata row groups contain 589,900 compressed bytes across
  the source host, format, publication year and split columns combined. Query these columns first
  and fetch text for selected keys. The sampled predicates overlap all five metadata row groups.
  Column projection is the immediate saving. Benchmark sorting or coarse partitioning only if
  repeated filtering becomes expensive. [DuckDB Parquet documentation](https://duckdb.org/docs/stable/data/parquet/overview)
- **Identity lookups.** In a sampled query plan, the existing DuckDB primary key used an index
  scan for a native identity lookup. A point subquery on the unique ID followed by formal-status
  and body SHA checks kept that plan. Joining on batches of keys is a separate workload. Published
  release files have 150 disjoint, contiguous row-number ranges. The 48 derived paragraph files
  have overlapping ranges. If measured derived point reads become expensive, evaluate a narrow
  versioned identity-to-file locator or a range-sorted Spark copy.
  [DuckDB index documentation](https://duckdb.org/docs/stable/guides/performance/indexing)
- **Near-duplicate retrieval.** The 977 capped buckets contained 80,789 of 7,214,088 postings.
  Capping reduced theoretical raw within-bucket pair occurrences before the other filters by
  85.43%. This is a candidate-volume comparison. The verifier scored 17,191 distinct pairs and
  4,504 met the exact shingle threshold. Candidate generation may miss dense reuse. Compare
  caps of 32 and 64 on a fixed review sample before any change. The candidate pairs involved
  18,317 distinct endpoints, so computing each endpoint's shingle set once could reduce repeated
  derivations. Measure the added Spark shuffle and storage before adopting this.
  [Bottom-k sampling paper](https://arxiv.org/abs/1303.5479)


## What the first three notebooks establish

The first figure links the verified synthetic fixture in Notebooks 01 and 02 to the date evidence
in the formal release. The second shows how format-specific decisions and duplicate checks create
reviewable counts. The third shows why coarse paragraph-based chunking in Notebook 04 should
be measured by format.

Run and save Notebooks 01 and 02 to refresh their local fixture evidence. These cells use the
current Notebook 03 output, or the published full snapshot when run separately. Each figure is
also saved as a PNG beside the snapshot in its figures directory.


Set FOT_STRUCTURE_FIGURES=1 before running this optional section.

The saved figure previews show the published core snapshot. Rerun the optional cells to refresh them.

In [ ]:
if os.environ.get("FOT_STRUCTURE_FIGURES") == "1":
    import ast
    import json
    from collections import Counter
    from hashlib import file_digest, sha256

    import matplotlib.pyplot as plt
    import nbformat
    from matplotlib.ticker import EngFormatter, LogLocator, NullLocator, StrMethodFormatter

    from fear_temperature.config import PROJECT_ROOT

    # Load the evidence written by the three notebook stages.
    structure_root = globals().get(
        "OUTPUT", PROJECT_ROOT / "work_packages/formal_corpus_release_20261117/structure-v2"
    )
    audit = json.loads((structure_root / "STRUCTURE_AUDIT.json").read_text(encoding="utf-8"))

    def saved_result(filename, cell_id, output_type):
        notebook = nbformat.read(PROJECT_ROOT / "notebooks" / filename, as_version=4)
        for cell in notebook.cells:
            if cell.id == cell_id:
                for output in cell.outputs:
                    if output.output_type == output_type:
                        payload = (
                            output.text if output_type == "stream" else output.data["text/plain"]
                        )
                        return ast.literal_eval(payload)
        raise ValueError(f"Run and save {filename} to record its verified result.")

    fixture_check = saved_result("01_workspace_check.ipynb", "fixture-check", "execute_result")
    fixture_run = saved_result("02_synthetic_end_to_end.ipynb", "inspect-tables", "stream")
    fixture_manifest_path = PROJECT_ROOT / fixture_run["run"] / "manifest.json"
    if not fixture_manifest_path.is_file():
        raise FileNotFoundError("Run and save Notebook 02 to create the local fixture manifest.")
    fixture_manifest = json.loads(fixture_manifest_path.read_text(encoding="utf-8"))
    fixture_name = "data/fixtures/synthetic_passages.jsonl"
    fixture_bytes = (PROJECT_ROOT / fixture_name).read_bytes()
    fixture_hash = sha256(fixture_bytes).hexdigest()
    if (
        fixture_check["synthetic_fixture"] != "PASS"
        or fixture_hash != fixture_check["sha256"]
        or fixture_hash != fixture_manifest["inputs"][fixture_name]
        or fixture_manifest["status"] != "completed"
        or fixture_manifest["mode"] != "synthetic_fixture"
        or fixture_manifest["run_id"] != fixture_manifest_path.parent.name
        or fixture_manifest["counts"] != fixture_run["counts"]
    ):
        raise ValueError("The fixture evidence from Notebooks 01 and 02 does not reconcile.")
    with (structure_root / "MANIFEST.json").open("rb") as handle:
        if file_digest(handle, "sha256").hexdigest() != audit["manifest_sha256"]:
            raise ValueError("Notebook 03 audit does not match the structure manifest.")

    role_counts = Counter(json.loads(line)["publisher_role"] for line in fixture_bytes.splitlines())
    fixture_counts = fixture_manifest["counts"]
    dates = audit["normalization"]["precision_counts"]
    if (
        sum(role_counts.values()) != fixture_check["input_records"]
        or sum(dates.values()) != audit["selected_records"]
    ):
        raise ValueError("The plotted populations do not reconcile.")

    scope = "Full formal release" if audit["run_mode"] == "full" else "Smoke subset"
    figure_dir = structure_root.with_name(f"{structure_root.name}-figures")
    figure_dir.mkdir(exist_ok=True)
    navy = "#142C3C"
    teal = "#007E77"
    blue = "#3B617F"
    amber = "#AB5A20"
    muted = "#516574"
    grid = "#DCE5E9"

    def format_axis(ax, orientation):
        ax.set_axisbelow(True)
        ax.grid(axis=orientation, color=grid, linewidth=0.8)
        ax.tick_params(length=0, labelsize=10, colors=muted)
        ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

    def finish_plot(fig, filename):
        path = figure_dir / filename
        fig.savefig(path, dpi=200, facecolor="white")
        plt.show()
        plt.close(fig)
        print(path.relative_to(PROJECT_ROOT).as_posix())

    fig, (a, b, c) = plt.subplots(
        1, 3, figsize=(15.6, 6.1), gridspec_kw={"width_ratios": [1, 1.13, 1.25]}
    )
    fig.subplots_adjust(left=0.07, right=0.96, top=0.73, bottom=0.26, wspace=0.52)
    fig.text(
        0.055,
        0.91,
        "From fixture checks to corpus structure",
        fontsize=22,
        weight="bold",
        color=navy,
    )
    fig.text(
        0.055,
        0.845,
        f"Synthetic interface checks in 01 and 02   |   "
        f"03 {scope.lower()} · {audit['selected_records']:,} records",
        fontsize=12,
        color=muted,
    )

    a.set_title("01  Verify the input", loc="left", y=1.08, fontsize=14, weight="bold", color=navy)
    roles = sorted(role_counts)
    bars = a.bar(
        [role.title() for role in roles],
        [role_counts[role] for role in roles],
        color=blue,
        width=0.59,
    )
    a.bar_label(bars, padding=5, color=navy, fontsize=11)
    a.set_ylim(0, max(role_counts.values()) * 1.22)
    a.set_ylabel("Synthetic input records", fontsize=10, color=muted)
    format_axis(a, "y")
    a.text(
        0,
        -0.23,
        f"{sum(role_counts.values()):,} rows passed schema and fixture hash checks.\n"
        f"Python environment and package identity were checked.",
        transform=a.transAxes,
        va="top",
        fontsize=9.5,
        color=muted,
        linespacing=1.55,
    )

    b.set_title(
        "02  Verify lineage and retrieval",
        loc="left",
        y=1.08,
        fontsize=14,
        weight="bold",
        color=navy,
    )
    bars = b.bar(
        ["Input\nrows", "Canonical\npassages", "Lineage\nrows"],
        [fixture_counts[key] for key in ("input", "canonical_passages", "lineage")],
        color=[blue, teal, blue],
        width=0.58,
    )
    b.bar_label(bars, padding=5, color=navy, fontsize=11)
    b.set_ylim(0, fixture_counts["input"] * 1.22)
    b.set_ylabel("Synthetic fixture counts", fontsize=10, color=muted)
    format_axis(b, "y")
    retrieved, relevant, future_worry = fixture_manifest["tfidf_counts"]
    b.text(
        0,
        -0.23,
        f"{fixture_counts['duplicates']:,} exact copy retains its lineage.\n"
        f"TF-IDF top {retrieved} · {relevant} relevant, {future_worry} tagged future worry.\n"
        f"{fixture_manifest['role_time_rows']:,} role-month rows and "
        f"{len(fixture_manifest['files']):,} files verified.",
        transform=b.transAxes,
        va="top",
        fontsize=9.5,
        color=muted,
        linespacing=1.55,
    )

    c.set_title(
        "03  Preserve date precision", loc="left", y=1.08, fontsize=14, weight="bold", color=navy
    )
    date_order = ["day", "month", "year", "year_range", "instant", "unknown"]
    categories = [key for key in date_order if dates.get(key, 0)]
    if set(categories) != set(dates):
        raise ValueError("The audit includes an unrecognised date precision.")
    date_counts = [dates[key] for key in categories]
    c.set_xscale("log")
    c.set_xlim(1, max(date_counts) * 13)
    c.xaxis.set_major_locator(LogLocator(base=10, numticks=8))
    c.xaxis.set_major_formatter(EngFormatter(sep=""))
    c.xaxis.set_minor_locator(NullLocator())
    for y, count in enumerate(date_counts):
        c.hlines(y, 1, count, color=teal, linewidth=2, alpha=0.65)
        c.scatter(count, y, color=teal, s=43, zorder=3)
        c.annotate(
            f"{count:,}",
            (count, y),
            xytext=(7, 0),
            textcoords="offset points",
            va="center",
            fontsize=10,
            color=navy,
        )
    c.set_yticks(range(len(categories)), [key.replace("_", " ").capitalize() for key in categories])
    c.set_ylim(len(categories) - 0.4, -0.6)
    c.set_xlabel("Formal records (log scale)", color=muted, fontsize=10)
    format_axis(c, "x")
    c.tick_params(axis="x", labelsize=9)
    time_evidence = audit["date_semantics"]
    c.text(
        0,
        -0.23,
        f"{time_evidence['exact_publication_utc']:,} records carry an exact UTC instant.\n"
        f"UTC and source dates differ for {time_evidence['source_and_utc_day_differ']:,} records.\n"
        f"The calendars assign different years to {time_evidence['source_and_utc_year_differ']:,}.",
        transform=c.transAxes,
        va="top",
        fontsize=9.5,
        color=muted,
        linespacing=1.55,
    )

    fig.text(
        0.055,
        0.035,
        "Sources  ·  Verified synthetic fixture (01 and 02) and structure audit (03).",
        color=muted,
        fontsize=9,
    )
    finish_plot(fig, "01_foundation_workflow.png")

In [ ]:
if os.environ.get("FOT_STRUCTURE_FIGURES") == "1":
    from matplotlib.ticker import MaxNLocator

    fuzzy = audit["fuzzy_search"]
    exact = audit["exact_reuse"].get("release_clean_body", {})
    rules = {row["rule"]: row["removed_spans"] for row in audit["cleaning_by_rule"]}
    signatures = rules.get("forum_tail_device_signature", 0)
    quote_blocks = rules.get("forum_bbcode_quote", 0) + rules.get("forum_html_blockquote", 0)
    formats = [
        row
        for row in audit["cleaning_by_format"]
        if row["edited_records"] or row["records_with_review_flags"]
    ]
    formats.sort(
        key=lambda row: (row["edited_records"] + row["records_with_review_flags"], row["format"]),
        reverse=True,
    )
    fig, (edits, overlap) = plt.subplots(
        1, 2, figsize=(15.4, 7.9), gridspec_kw={"width_ratios": [1.1, 1]}
    )
    fig.subplots_adjust(left=0.175, right=0.94, top=0.74, bottom=0.29, wspace=0.47)
    fig.text(
        0.055,
        0.93,
        "What structure normalization makes measurable",
        fontsize=22,
        weight="bold",
        color=navy,
    )
    fig.text(
        0.055,
        0.88,
        f"{scope} · {audit['selected_records']:,} records · "
        f"{audit['total_paragraphs']:,} paragraph spans "
        f"· {len(audit['cleaning_by_format']):,} observed format labels",
        fontsize=12,
        color=muted,
    )

    edits.set_title(
        "Cleaning and review by format", loc="left", y=1.12, fontsize=14, weight="bold", color=navy
    )
    for delta, key, label, color in [
        (-0.18, "edited_records", "Structure edited", teal),
        (0.18, "records_with_review_flags", "Flagged for review", amber),
    ]:
        counts = [row[key] for row in formats]
        bars = edits.barh(
            [index + delta for index in range(len(formats))],
            counts,
            height=0.32,
            color=color,
            label=label,
        )
        edits.bar_label(
            bars,
            labels=[f"{count:,}" if count else "" for count in counts],
            padding=5,
            fontsize=10,
            color=navy,
        )
    edits.set_yticks(
        range(len(formats)),
        [f"{row['format']}  ({row['records']:,})" for row in formats],
    )
    if formats:
        edits.set_ylim(len(formats) - 0.45, -0.55)
        edits.set_xlim(
            0,
            max(max(row["edited_records"], row["records_with_review_flags"]) for row in formats)
            * 1.28,
        )
    edits.set_xlabel("Affected records", color=muted, fontsize=10, labelpad=8)
    edits.xaxis.set_major_locator(MaxNLocator(5, integer=True))
    edits.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    edits.legend(loc="upper left", bbox_to_anchor=(0, 1.08), ncol=2, frameon=False, fontsize=9.5)
    format_axis(edits, "x")
    edits.text(
        0,
        -0.20,
        f"{audit['edited_records']:,} edited · {audit['records_with_review_flags']:,} flagged.\n"
        f"Forum removals included {signatures:,} signatures and {quote_blocks:,} quote blocks.\n"
        "Parentheses show format totals. Bars can overlap.",
        transform=edits.transAxes,
        va="top",
        fontsize=10,
        color=muted,
        linespacing=1.55,
    )

    overlap.set_title(
        "Near-duplicate evidence", loc="left", y=1.12, fontsize=14, weight="bold", color=navy
    )
    stages = [
        ("Candidates\nverified", fuzzy["candidate_pairs_verified"], blue),
        ("Matches", fuzzy["matched_pairs"], teal),
        ("Cross-split\nmatches", fuzzy["cross_split_pairs"], amber),
    ]
    bars = overlap.barh(
        [name for name, _, _ in stages],
        [count for _, count, _ in stages],
        color=[color for _, _, color in stages],
        height=0.49,
    )
    overlap.bar_label(
        bars, labels=[f"{count:,}" for _, count, _ in stages], padding=6, color=navy, fontsize=11
    )
    overlap.set_xlim(0, max(fuzzy["candidate_pairs_verified"], 1) * 1.24)
    overlap.invert_yaxis()
    overlap.set_xlabel("Distinct-body pairs", color=muted, fontsize=10, labelpad=8)
    overlap.xaxis.set_major_locator(MaxNLocator(5, integer=True))
    overlap.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    format_axis(overlap, "x")
    overlap.text(
        0,
        -0.20,
        f"{fuzzy['cross_split_affected_records']:,} records are linked to cross-split matches.\n"
        f"Exact bodies account for {exact.get('extra_members', 0):,} extra copies, with "
        f"{exact.get('cross_split_keys', 0):,} hashes spanning splits.\n"
        f"Near-copy rule uses {fuzzy['shingle_width']}-word shingles "
        f"with Jaccard ≥ {fuzzy['min_jaccard']:.2f}.",
        transform=overlap.transAxes,
        va="top",
        fontsize=10,
        color=muted,
        linespacing=1.55,
    )

    fig.text(
        0.055,
        0.043,
        f"Search coverage  ·  {fuzzy['below_fuzzy_evidence_minimum']:,} bodies fell below the "
        f"minimum shingle support. The search skipped {fuzzy['suppressed_large_buckets']:,} "
        "over-cap buckets.\n"
        "The Jaccard rule can miss containment-only copies. Counts show detected evidence.",
        fontsize=9.5,
        color=muted,
        linespacing=1.5,
    )
    finish_plot(fig, "02_structure_and_overlap.png")

In [ ]:
if os.environ.get("FOT_STRUCTURE_FIGURES") == "1":
    per_format = sorted(
        audit["cleaning_by_format"],
        key=lambda row: row["paragraphs"] / row["records"],
        reverse=True,
    )
    means = [row["paragraphs"] / row["records"] for row in per_format]
    labels = [f"{row['format']}  ({row['records']:,})" for row in per_format]
    fig, ax = plt.subplots(figsize=(12, 7.5))
    fig.subplots_adjust(left=0.25, right=0.93, top=0.78, bottom=0.16)
    fig.text(
        0.055, 0.925, "Paragraph structure varies by format", fontsize=22, weight="bold", color=navy
    )
    fig.text(
        0.055,
        0.872,
        f"{scope} · {audit['selected_records']:,} records",
        fontsize=12,
        color=muted,
    )
    bars = ax.barh(labels, means, height=0.60, color=teal)
    ax.bar_label(bars, labels=[f"{mean:.1f}" for mean in means], padding=5, color=navy, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, max(means) * 1.14)
    ax.set_xlabel("Mean paragraph spans per record", color=muted, fontsize=11, labelpad=8)
    format_axis(ax, "x")
    fig.text(
        0.055,
        0.04,
        "Parentheses show the format totals. Means use the blank-line spans recorded in 03.\n"
        "Notebook 04 measures chunk size and coverage by format.",
        fontsize=9.5,
        color=muted,
        linespacing=1.5,
    )
    finish_plot(fig, "03_paragraphs_by_format.png")